C28. Где именно рождается эмерджентность: разложение по осям рубрики.....


In [1]:
import sys, os

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/panel_mechanisms')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
import statsmodels.api as sm
A = load_arms()
ITEMS, POS = load_items()
print()
AX_ORDER = ['accuracy', 'completeness', 'reasoning', 'urgency', 'uncertainty', 'communication_quality']
REF = list(ITEMS)[0]
INFO = pd.DataFrame([{'case': k[0], 'cid': k[1], 'points': ITEMS[REF][k]['points'], 'axis': ITEMS[REF][k]['axis']} for k in POS])
show(INFO.groupby('axis').agg(пунктов=('points', 'size'), сумма_баллов=('points', 'sum'), ср_вес=('points', 'mean')).assign(доля_веса=lambda x: x['сумма_баллов'] / x['сумма_баллов'].sum()).reindex(AX_ORDER), 'Структура рубрики по осям')


плеч загружено: 18
   baseline gpt, baseline qwen, baseline deepseekv4, baseline deepseekv4pro, baseline gemini, baseline glm, baseline kimi, USC@3 gpt, USC@5 gpt, USC@7 gpt, homog gpt, homog qwen, homog gemini, homog deepseekv4, P-weak, P-mixed, P-strong, P-base


плеч с попунктными вердиктами: 18
общих положительных пунктов: 2249


Структура рубрики по осям
------------------------------------------------------------
                       пунктов  сумма_баллов  ср_вес  доля_веса
axis                                                           
accuracy                   536          4111  7.6698     0.2794
completeness               768          4711  6.1341     0.3201
reasoning                  234          1635  6.9872     0.1111
urgency                    251          1833  7.3028     0.1246
uncertainty                261          1446  5.5402     0.0983
communication_quality      199           980  4.9246     0.0666


In [2]:
def axis_scores(arm):
    f = ITEMS[arm]
    rows = [{'axis': ITEMS[REF][k]['axis'], 'points': ITEMS[REF][k]['points'], 'met': int(f[k]['met']), 'case': k[0]} for k in POS]
    D = pd.DataFrame(rows)
    g = D.groupby('axis').apply(lambda x: (x['met'] * x['points']).sum() / x['points'].sum(), include_groups=False)
    return g
AS = pd.DataFrame({a: axis_scores(a) for a in ITEMS}).T
AS = AS[[c for c in AX_ORDER if c in AS.columns]]
show(AS.sort_values('accuracy', ascending=False), 'Доля выполненных пунктов по осям')



Доля выполненных пунктов по осям
------------------------------------------------------------
axis                    accuracy  completeness  reasoning  urgency  uncertainty  communication_quality
P-strong                  0.9175        0.7103     0.9388   0.9520       0.6307                 0.5918
P-mixed                   0.9127        0.6646     0.9242   0.9416       0.5878                 0.6551
homog deepseekv4          0.9017        0.6476     0.9205   0.9351       0.5809                 0.6959
P-base                    0.8903        0.7228     0.8991   0.8963       0.5567                 0.6429
P-weak                    0.8779        0.6213     0.9291   0.8871       0.5297                 0.6102
homog gemini              0.8752        0.5793     0.8972   0.8456       0.4620                 0.5837
homog gpt                 0.8606        0.6829     0.8789   0.9002       0.5906                 0.5286
homog qwen                0.8502        0.5302     0.8612   0.8500       0.5367  

In [3]:
W = INFO.groupby('axis')['points'].sum()
W = (W / W.sum()).reindex(AS.columns)

def decompose(a, b):
    diff = AS.loc[a] - AS.loc[b]
    contrib = diff * W
    out = pd.DataFrame({'доля выполненных, A': AS.loc[a], 'B': AS.loc[b], 'разница по оси': diff, 'вес оси': W, 'вклад в общую разницу': contrib})
    out['доля вклада, %'] = 100 * contrib / contrib.sum()
    return out.sort_values('вклад в общую разницу', ascending=False)
for a, b in [('P-base', 'baseline gpt'), ('P-base', 'USC@7 gpt'), ('homog gpt', 'baseline gpt')]:
    if a in AS.index and b in AS.index:
        D = decompose(a, b)
        show(D, 'Разложение «%s − %s» по осям (сумма вкладов = %.4f)' % (a, b, D['вклад в общую разницу'].sum()))



Разложение «P-base − baseline gpt» по осям (сумма вкладов = 0.1516)
-------------------------------------------------------------------
                       доля выполненных, A      B  разница по оси  вес оси  вклад в общую разницу  доля вклада, %
axis                                                                                                             
completeness                        0.7228 0.5194          0.2034   0.3201                 0.0651         42.9404
uncertainty                         0.5567 0.2586          0.2981   0.0983                 0.0293         19.3187
urgency                             0.8963 0.7741          0.1222   0.1246                 0.0152         10.0403
reasoning                           0.8991 0.7639          0.1352   0.1111                 0.0150          9.9059
accuracy                            0.8903 0.8392          0.0511   0.2794                 0.0143          9.4128
communication_quality               0.6429 0.4520          0.1908

In [4]:
def member_axis(pname, how='mean'):
    members = ['baseline ' + m for m in COMPOSITION[pname]]
    members = [m for m in members if m in ITEMS]
    if not members:
        return None
    rows = []
    for k in POS:
        v = [int(ITEMS[m][k]['met']) for m in members]
        rows.append({'axis': ITEMS[REF][k]['axis'], 'points': ITEMS[REF][k]['points'], 'val': np.mean(v) if how == 'mean' else max(v)})
    D = pd.DataFrame(rows)
    return D.groupby('axis').apply(lambda x: (x['val'] * x['points']).sum() / x['points'].sum(), include_groups=False)
rows = []
for p in PANELS:
    if p not in AS.index:
        continue
    mm = member_axis(p, 'mean')
    mx = member_axis(p, 'max')
    if mm is None:
        continue
    for ax in AS.columns:
        rows.append({'панель': p, 'ось': ax, 'панель_знач': AS.loc[p, ax], 'среднее состава': mm.get(ax, np.nan), 'максимум состава': mx.get(ax, np.nan), 'Δ к среднему': AS.loc[p, ax] - mm.get(ax, np.nan), 'Δ к максимуму': AS.loc[p, ax] - mx.get(ax, np.nan), 'вес оси': W.get(ax, np.nan)})
PA = pd.DataFrame(rows)
show(PA.pivot_table(index='ось', values=['Δ к среднему', 'Δ к максимуму'], aggfunc='mean').reindex(AX_ORDER), 'Прирост панели по осям (среднее по 8 панелям)')
print()
print('Ключевая колонка — «Δ к максимуму». Положительное значение означает,')
print('что панель на этой оси превосходит ЛУЧШЕГО из своих участников,')
print('то есть добавляет то, чего не было ни у кого.')



Прирост панели по осям (среднее по 8 панелям)
------------------------------------------------------------
                       Δ к максимуму  Δ к среднему
ось                                               
accuracy                      0.0432        0.0958
completeness                  0.1322        0.2326
reasoning                     0.0924        0.1478
urgency                       0.0891        0.1619
uncertainty                   0.2495        0.3119
communication_quality         0.0716        0.1601

Ключевая колонка — «Δ к максимуму». Положительное значение означает,
что панель на этой оси превосходит ЛУЧШЕГО из своих участников,
то есть добавляет то, чего не было ни у кого.


In [5]:
rows = []
for p in PANELS:
    if p not in ITEMS:
        continue
    members = ['baseline ' + m for m in COMPOSITION[p]]
    members = [m for m in members if m in ITEMS]
    if not members:
        continue
    for k in POS:
        rows.append({'panel': p, 'case': k[0], 'cid': k[1], 'axis': ITEMS[REF][k]['axis'], 'points': ITEMS[REF][k]['points'], 'is_panel': 1, 'met': int(ITEMS[p][k]['met'])})
        rows.append({'panel': p, 'case': k[0], 'cid': k[1], 'axis': ITEMS[REF][k]['axis'], 'points': ITEMS[REF][k]['points'], 'is_panel': 0, 'met': int(max((int(ITEMS[m][k]['met']) for m in members)))})
L = pd.DataFrame(rows)
print('наблюдений (пункт × панель × {панель, максимум состава}):', len(L))
X = pd.get_dummies(L['axis'], prefix='ax', drop_first=True).astype(float)
inter = X.mul(L['is_panel'], axis=0)
inter.columns = [c + ':панель' for c in inter.columns]
XX = sm.add_constant(pd.concat([L[['is_panel']].astype(float), X, inter], axis=1))
m = sm.Logit(L['met'].astype(float), XX).fit(disp=0, cov_type='cluster', cov_kwds={'groups': L['case'].astype('category').cat.codes})
res = pd.DataFrame({'коэф': m.params, 'SE': m.bse, 'z': m.tvalues, 'p': m.pvalues, 'OR': np.exp(m.params)})
show(res[res.index.str.contains('панель')], 'Логистическая модель: превосходство панели над максимумом состава по осям')
print()
base_ax = sorted(L['axis'].dropna().unique())[0]
print('База сравнения — ось «%s». Коэффициент «is_panel» показывает эффект' % base_ax)
print('на базовой оси, а члены «ось:панель» — насколько он отличается на')
print('остальных осях.')
print()
print('псевдо-R² = %.4f' % m.prsquared)


наблюдений (пункт × панель × {панель, максимум состава}): 35984

Логистическая модель: превосходство панели над максимумом состава по осям
-------------------------------------------------------------------------
                                   коэф     SE       z      p     OR
ax_communication_quality:панель -0.0778 0.0873 -0.8907 0.3731 0.9252
ax_completeness:панель           0.1901 0.0659  2.8870 0.0039 1.2094
ax_reasoning:панель              0.4581 0.1037  4.4182 0.0000 1.5811
ax_uncertainty:панель            0.6974 0.0895  7.7935 0.0000 2.0086
ax_urgency:панель                0.3924 0.1078  3.6382 0.0003 1.4805

База сравнения — ось «accuracy». Коэффициент «is_panel» показывает эффект
на базовой оси, а члены «ось:панель» — насколько он отличается на
остальных осях.



псевдо-R² = 0.1109


In [6]:
if 'P-base' in AS.index and 'baseline gpt' in AS.index:
    d = decompose('P-base', 'baseline gpt')
    rank_here = d['разница по оси'].rank(ascending=False)
    print('Порядок осей по величине прироста P-base над baseline gpt:')
    for i, (ax, v) in enumerate(d['разница по оси'].sort_values(ascending=False).items(), 1):
        print('   %d. %-24s %+.4f  (вклад в итог %.1f%%)' % (i, ax, v, d.loc[ax, 'доля вклада, %']))
    print()
    print('§5 работы утверждает: максимальный прирост на uncertainty и urgency,')
    print('accuracy растёт слабее всех. Сверьте с порядком выше.')


Порядок осей по величине прироста P-base над baseline gpt:
   1. uncertainty              +0.2981  (вклад в итог 19.3%)
   2. completeness             +0.2034  (вклад в итог 42.9%)
   3. communication_quality    +0.1908  (вклад в итог 8.4%)
   4. reasoning                +0.1352  (вклад в итог 9.9%)
   5. urgency                  +0.1222  (вклад в итог 10.0%)
   6. accuracy                 +0.0511  (вклад в итог 9.4%)

§5 работы утверждает: максимальный прирост на uncertainty и urgency,
accuracy растёт слабее всех. Сверьте с порядком выше.


In [7]:
if 'P-base' in AS.index and 'baseline gpt' in AS.index:
    d = decompose('P-base', 'baseline gpt')
    tot_panel = 26746.0
    tot_base = 1752.0
    extra = tot_panel - tot_base
    cost = d[['вклад в общую разницу', 'доля вклада, %']].copy()
    cost['токенов на ось'] = extra * cost['доля вклада, %'] / 100
    cost['токенов на 0.01 балла'] = cost['токенов на ось'] / (cost['вклад в общую разницу'] * 100).replace(0, np.nan)
    show(cost, 'Во что обходится прирост по каждой оси')
    print()
    print('ВНИМАНИЕ: распределение бюджета по осям здесь условное. Токены')
    print('тратятся на порождение текста целиком, а не на отдельные оси.')
    print('Таблица показывает не причинную стоимость, а «цену за результат».')



Во что обходится прирост по каждой оси
------------------------------------------------------------
                       вклад в общую разницу  доля вклада, %  токенов на ось  токенов на 0.01 балла
axis                                                                                               
completeness                          0.0651         42.9404      10732.5199              1648.6405
uncertainty                           0.0293         19.3187       4828.5137              1648.6405
urgency                               0.0152         10.0403       2509.4827              1648.6405
reasoning                             0.0150          9.9059       2475.8736              1648.6405
accuracy                              0.0143          9.4128       2352.6401              1648.6405
communication_quality                 0.0127          8.3819       2094.9700              1648.6405

ВНИМАНИЕ: распределение бюджета по осям здесь условное. Токены
тратятся на порождение текста целик

In [8]:
save(AS.reset_index(), 'criterion_recovery_by_rubric_axis_axis_scores.csv')
save(PA, 'criterion_recovery_by_rubric_axis_panel_vs_members.csv')
save(res.reset_index(), 'criterion_recovery_by_rubric_axis_logit.csv')
if 'P-base' in AS.index:
    save(decompose('P-base', 'baseline gpt').reset_index(), 'criterion_recovery_by_rubric_axis_decomposition.csv')


сохранено: out/criterion_recovery_by_rubric_axis_axis_scores.csv  (18 строк)
сохранено: out/criterion_recovery_by_rubric_axis_panel_vs_members.csv  (48 строк)
сохранено: out/criterion_recovery_by_rubric_axis_logit.csv  (12 строк)
сохранено: out/criterion_recovery_by_rubric_axis_decomposition.csv  (6 строк)
